In [ ]:
import sys
sys.path.append("../")

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import yaml

from npu_yolov8n import get_dataloader
from npu_yolov8n.data import preprocess_dataset

from npu_yolov8n import load_model, load_QAT_model, load_NPU_model

from npu_yolov8n import load_model
from npu_yolov8n.utils import ActivationStatsCollector
from npu_yolov8n.models.blocks.qat_blocks import QConv
from npu_yolov8n.models.blocks.basic_blocks import DFL

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# load dataset 
train_loader, val_loader, coco_id2label, label2coco_category = get_dataloader(data_root='../datasets/coco')

# load quantized model
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

# Activation and Weight Analysis

## Activation

In [ ]:
A_NUM_BITS = 8 
num_bits = A_NUM_BITS - 1 # -1 for signed activations

asc = ActivationStatsCollector(Qmodel, (QConv, DFL))

run_batch = 2 # number of calibration batches to run NOTE: increase this for better accuracy
train_iter = iter(train_loader)
for _ in range(run_batch): # run multiple iterations to collect activation statistics
    with torch.no_grad():
        images, annotations = next(train_iter)
        inputs, batch = preprocess_dataset(images, annotations, coco_id2label, device) # preprocess dataset
        outputs = Qmodel(inputs, inference=True)
activation_stats = asc.get_stats()

act_stats = {}
for layer_name, acts in activation_stats.items():
    act_stats[layer_name] = torch.cat(acts, dim=0)  # concatenate all batches

import math
def get_frac_bits(num_ubits, x) -> int:
    exp = math.floor(math.log2((2**num_ubits - 1)/x)) 
    return exp 

af_list = []
names = list(act_stats.keys())
for i, name in enumerate(names):
    tensor = act_stats[name].flatten()
    max_val = tensor.kthvalue(int(tensor.numel() * 0.999)).values.item()
    f = get_frac_bits(num_bits, max_val)
    af_list.append(f)
    
    print(f"{name:<25} | max={max_val:<7.2f} | frac_bits={f}")
    


## Weight

In [ ]:
W_NUM_BITS = 8

# load quantized model
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']
Qmodel = load_model(checkpoint_path, device, model_type='qat', model_qcfg=loaded_cfg)

w_stats = Qmodel.state_dict()

names = list(w_stats.keys())

wf_list = []

for i, name in enumerate(names):
    if 'weight' not in name:
        continue
    tensor = w_stats[name].flatten()
    max_val = tensor.kthvalue(int(tensor.numel() * 0.999)).values.item()
    f = get_frac_bits(W_NUM_BITS-1, max_val)
    wf_list.append(f)

    print(f"{name:<35} | max={max_val:<7.2f} | frac_bits={f}")

## QAT Config Gen

In [ ]:
import copy


def bottleneck_qcfg_gen(cfg):
    return {
        "cv1": cfg.pop(0),
        "cv2": cfg.pop(0),
    }

def c2f_qcfg_gen(cfg, n):
    ret = {
        "cv1": cfg.pop(0),
        "cv2": cfg.pop(0),
    }
    for i in range(n):
        ret[f"m.{i}"] = bottleneck_qcfg_gen(cfg)
    return ret

def sppf_qcfg_gen(cfg):
    return {
        "cv1": cfg.pop(0),
        "cv2": cfg.pop(0),
    }

def detect_qcfg_gen(cfg):
    return {
        "cv2.0": cfg.pop(0),
        "cv2.1": cfg.pop(0),
        "cv2.2": cfg.pop(0),

        "cv3.0": cfg.pop(0),
        "cv3.1": cfg.pop(0),
        "cv3.2": cfg.pop(0),
    }

a = copy.deepcopy(af_list) # activation fraction bits, sequential
w = copy.deepcopy(wf_list) # weight fraction bits, sequential
if W_NUM_BITS != A_NUM_BITS:
    print("Warning: W_NUM_BITS and A_NUM_BITS are different. Using this configuration, NPU model is not supported.")
    num_bits = [W_NUM_BITS, A_NUM_BITS]
else:
    num_bits = A_NUM_BITS

# note: config format: [num_bits, w_fraction_bits, a_fraction_bits]
quantization_config = {
    "layer_0": [num_bits, w.pop(0), a.pop(0)],
    "layer_1": [num_bits, w.pop(0), a.pop(0)],
    "layer_2": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_3": [num_bits, w.pop(0), a.pop(0)],
    "layer_4": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 1.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 1.cv2
    ], 2),
    "layer_5": [num_bits, w.pop(0), a.pop(0)],
    "layer_6": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 1.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 1.cv2
    ], 2),
    "layer_7": [num_bits, w.pop(0), a.pop(0)],
    "layer_8": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_9": sppf_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
    ]),
    "layer_12": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_15": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_16": [num_bits, w.pop(0), a.pop(0)],
    "layer_18": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_19": [num_bits, w.pop(0), a.pop(0)],
    "layer_21": c2f_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv1
        [num_bits, w.pop(0), a.pop(0)], # cv2
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv1
        [num_bits, w.pop(0), a.pop(0)], # bottleneck 0.cv2
    ], 1),
    "layer_22": detect_qcfg_gen([
        [num_bits, w.pop(0), a.pop(0)], # cv2.0
        [num_bits, w.pop(0), a.pop(0)], # cv2.1
        [num_bits, w.pop(0), a.pop(0)], # cv2.2
        [num_bits, w.pop(0), a.pop(0)], # cv3.0
        [num_bits, w.pop(0), a.pop(0)], # cv3.1
        [num_bits, w.pop(0), a.pop(0)], # cv3.2
    ]),
}

import yaml


# YAML save 
class MyDumper(yaml.SafeDumper):
    pass

def represent_list_flow(dumper, data):
    return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

MyDumper.add_representer(list, represent_list_flow)

from datetime import datetime

# Model Save
now = datetime.now()
formatted = now.strftime("%m%d%H%M")

name = 'qcfg_1'
# save
import os
os.makedirs('./config', exist_ok=True)

with open(f'./config/qcfg_{formatted}.yaml', 'w') as f:
    yaml.dump(quantization_config, f, Dumper=MyDumper, sort_keys=False)

# load
with open(f'./config/qcfg_{formatted}.yaml') as f:
    loaded_cfg = yaml.safe_load(f)

# check
loaded_cfg


# Fix QAT Config Compatible for NPU Model

In [ ]:
# load
with open(f'./config/qcfg_{formatted}.yaml') as f:
    loaded_cfg = yaml.safe_load(f)

# check loaded_cfg
Nmodel = load_NPU_model(checkpoint_path, device, loaded_cfg) # fix quantization config issue (no automatic correction now...) you have to set fraction bits same, small value
